# NammaSpace STEP 2 on Google Colab (free GPU)

Trains the 3D Gaussian splat from the zip made by STEP 1 and packs a result zip for STEP 3.
Run the cells one by one with Shift+Enter and wait for each to finish.

**How this works:** gsplat only ships ready-made GPU builds for Python 3.10 and PyTorch 2.4 or older, and this machine has newer versions. So the notebook builds a separate Python 3.10 environment with `uv` and installs PyTorch 2.4.1 and the ready-made gsplat 1.4.0 there. That avoids the long silent compile that stalled the earlier notebooks. The notebook's own Python is not touched.

## One-time setup
1. **Put the bundle in Google Drive.** drive.google.com > New > New folder named `namma`. Upload `vid_..._to_remote.zip` from `workspace/to_gpu` into it.
2. **Open this notebook in Colab.** colab.research.google.com > File > Upload notebook.
3. **Pick the GPU.** Menu **Runtime > Change runtime type > Hardware accelerator > T4 GPU > Save**. The GPU is free but not guaranteed.
4. Run the cells from the top.

Free Colab has about 12 GB of RAM and disconnects when idle, so keep the tab open. The result zip is saved to your Drive folder, so it survives a disconnect.

## 1. Check that a GPU is attached
If this prints no GPU table, fix the accelerator setting and run it again.

In [ ]:
!nvidia-smi

## 2. Connect Google Drive
A popup asks for permission. Accept it.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3. Create the Python 3.10 environment
Downloads Python 3.10 and sets up an isolated environment. Takes about a minute.

In [ ]:
import glob
import json
import os
import shutil
import signal
import subprocess
import sys
import time
import zipfile

BASE = '/content'
VENV = BASE + '/ns'
PY = VENV + '/bin/python'
NS = VENV + '/bin'
UV = [sys.executable, '-m', 'uv']


def run(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
    process.wait()
    assert process.returncode == 0, 'command failed with exit code ' + str(process.returncode)


run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
run(UV + ['venv', VENV, '--python', '3.10', '--seed', '--allow-existing'])
run([PY, '--version'])

## 4. Install PyTorch 2.4.1 (CUDA 12.1) into that environment
Takes a few minutes.

In [ ]:
run(UV + ['pip', 'install', '--python', PY, 'torch==2.4.1+cu121', 'torchvision==0.19.1+cu121', '--extra-index-url', 'https://download.pytorch.org/whl/cu121', '--index-strategy', 'unsafe-best-match'])

## 5. Install the ready-made gsplat 1.4.0 and nerfstudio 1.1.5
Same versions as the lab PC. If the gsplat file is not found, the message will say so; send it to me instead of continuing.

In [ ]:
run(UV + ['pip', 'install', '--python', PY, 'torch==2.4.1+cu121', 'torchvision==0.19.1+cu121', 'numpy==1.26.4', 'protobuf==3.20.3', 'gsplat==1.4.0+pt24cu121', 'nerfstudio==1.1.5', '--extra-index-url', 'https://download.pytorch.org/whl/cu121', '--find-links', 'https://docs.gsplat.studio/whl/gsplat/', '--index-strategy', 'unsafe-best-match'])

## 6. Check the install
Must show torch 2.4.1, GPU available True, gsplat 1.4.0 and a path for `ns-train`.

In [ ]:
check = "import torch, gsplat; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'gpu available', torch.cuda.is_available()); print('gsplat', gsplat.__version__)"
result = subprocess.run([PY, '-c', check], capture_output=True, text=True)
print(result.stdout)
print(result.stderr[-1500:])
print('ns-train  :', os.path.exists(NS + '/ns-train'))
print('ns-export :', os.path.exists(NS + '/ns-export'))
assert result.returncode == 0, 'torch or gsplat did not import, see the message above'
assert os.path.exists(NS + '/ns-train') and os.path.exists(NS + '/ns-export'), 'nerfstudio is not installed'

## 7. Test gsplat's GPU code on its own
A tiny render on the GPU. With the ready-made build it should take seconds. Limited to 15 minutes. If it fails or times out, stop here and send me the message.

In [ ]:
capability = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'], capture_output=True, text=True).stdout.split()[0]
gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.splitlines()[0]
ENV = dict(os.environ, PATH=NS + ':' + os.environ['PATH'], TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD='1', TORCH_CUDA_ARCH_LIST=capability)
print('GPU:', gpu_name, '| compute capability', capability)

kernel_test = '''
import time
import torch
from gsplat.rendering import rasterization

start = time.time()
device = 'cuda'
means = torch.rand(200, 3, device=device)
means[:, 2] += 2.0
quats = torch.nn.functional.normalize(torch.rand(200, 4, device=device), dim=-1)
scales = torch.full((200, 3), 0.05, device=device)
opacities = torch.full((200,), 0.8, device=device)
colors = torch.rand(200, 3, device=device)
viewmats = torch.eye(4, device=device)[None]
Ks = torch.tensor([[[100.0, 0.0, 64.0], [0.0, 100.0, 64.0], [0.0, 0.0, 1.0]]], device=device)
rendered, alphas, meta = rasterization(means, quats, scales, opacities, colors, viewmats, Ks, 128, 128)
torch.cuda.synchronize()
print('gsplat GPU code works, seconds taken:', round(time.time() - start))
'''
result = subprocess.run([PY, '-c', kernel_test], capture_output=True, text=True, timeout=900, env=ENV)
print(result.stdout)
print(result.stderr[-2500:])
assert result.returncode == 0, 'gsplat test failed, see the message above'

## 8. Settings
The defaults are the real run: 30000 steps takes about 25 minutes on a free T4 for a scene of roughly 950 frames (tested on Kaggle and Colab, both gave PSNR 29.1).

- `ITERS`: training steps. 30000 is the full run. Use 2000 for a quick test that proves the notebook works.
- `DOWNSCALE`: 2 uses the half-size images in the bundle and fits the 16 GB GPU.
- `CACHE`: keep `cpu`.
- `MAX_MINUTES`: safety limit. The watcher stops training after this long.

In [ ]:
DRIVE_FOLDER = '/content/drive/MyDrive/namma'
ITERS = 30000
DOWNSCALE = 2
CACHE = 'cpu'
PROFILE = 'default'
MAX_MINUTES = 120

## 9. Copy your bundle

In [ ]:
zips = sorted(glob.glob(DRIVE_FOLDER + '/*_to_remote.zip'))
assert zips, 'No *_to_remote.zip in ' + DRIVE_FOLDER + '. Check the folder name and that the zip is uploaded.'
print('using', zips[-1])
BUNDLE = BASE + '/bundle'
shutil.rmtree(BASE + '/unzipped', ignore_errors=True)
shutil.rmtree(BUNDLE, ignore_errors=True)
with zipfile.ZipFile(zips[-1]) as archive:
    archive.extractall(BASE + '/unzipped')
top = glob.glob(BASE + '/unzipped/*/')[0]
shutil.copytree(top, BUNDLE)
os.chdir(BUNDLE)
print(os.listdir(BUNDLE))

## 10. Check the data

In [ ]:
from PIL import Image, ImageChops, ImageStat

scene = json.load(open('scene.json'))['scene']
full = len(os.listdir('dataset/images'))
half = len(os.listdir('dataset/images_2'))
print('scene  :', scene)
print('frames :', full, 'full size,', half, 'half size')
assert full > 30 and full == half, 'image folders are missing or do not match'
assert os.path.isfile('dataset/colmap/sparse/0/images.bin'), 'dataset/colmap/sparse/0 is missing'
files = sorted(glob.glob('dataset/images/*.jpg'))
sample = files[::max(1, len(files) // 8)][:8]
spreads = []
for path in sample:
    red, green, blue = Image.open(path).convert('RGB').split()
    spreads.append(ImageStat.Stat(ImageChops.difference(red, blue)).mean[0])
average = sum(spreads) / len(spreads)
print('colour check: red-vs-blue difference', round(average, 1), '(grayscale would be 0.0)')
assert average >= 1.0, 'images are grayscale'

## 11. Start training
Starts training as its own background program and returns at once.

In [ ]:
help_text = subprocess.run([NS + '/ns-train', 'splatfacto', '--help'], capture_output=True, text=True, env=ENV).stdout
assert 'max-num-iterations' in help_text, 'this nerfstudio does not know splatfacto'
os.makedirs('logs', exist_ok=True)
command = [
    NS + '/ns-train', 'splatfacto',
    '--data', 'dataset',
    '--output-dir', 'outputs',
    '--experiment-name', scene,
    '--max-num-iterations', str(ITERS),
    '--steps-per-save', '2000',
    '--vis', 'tensorboard',
    '--pipeline.datamanager.cache-images', CACHE,
    'colmap', '--colmap-path', 'colmap/sparse/0', '--images-path', 'images', '--downscale-factor', str(DOWNSCALE),
]
print(' '.join(command))
train_started = time.time()
train_log = open('logs/train.log', 'w')
process = subprocess.Popen(['nice', '-n', '10'] + command, stdout=train_log, stderr=subprocess.STDOUT, text=True, env=ENV, start_new_session=True)
print('training started in the background, pid', process.pid)

## 12. Watch progress until training finishes
One line per minute: elapsed time, GPU use, the program using the most RAM, and the newest log line. Safe to stop and run again; training keeps going. Stops training by itself after `MAX_MINUTES`.

In [ ]:
def gpu_line():
    result = subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used', '--format=csv,noheader'], capture_output=True, text=True)
    lines = result.stdout.strip().splitlines()
    return lines[0] if lines else 'n/a'


def top_memory_line():
    result = subprocess.run(['ps', '-eo', 'rss,comm', '--sort=-rss'], capture_output=True, text=True)
    lines = result.stdout.strip().splitlines()
    if len(lines) < 2:
        return 'n/a'
    rss_kb, name = lines[1].split(None, 1)
    return name + ' ' + str(round(int(rss_kb) / 1e6, 1)) + ' GB'


while process.poll() is None:
    time.sleep(60)
    if time.time() - train_started > MAX_MINUTES * 60:
        os.killpg(process.pid, signal.SIGKILL)
        process.wait()
        print('STOPPED: time limit of', MAX_MINUTES, 'minutes reached')
        break
    lines = open('logs/train.log').read()[-600:].replace('\r', '\n').strip().splitlines()
    newest = lines[-1][:110] if lines else ''
    print(time.strftime('%H:%M:%S'), '|', int(time.time() - train_started) // 60, 'min | GPU', gpu_line(), '| top RAM', top_memory_line(), '|', newest)
train_seconds = int(time.time() - train_started)
print('exit code:', process.returncode)
if process.returncode != 0:
    print(open('logs/train.log').read()[-3000:])
assert process.returncode == 0, 'training failed, see the log above'
print('training finished in', train_seconds // 60, 'minutes')

## 12b. If training looks stuck: show where it is stuck
Only run this if the GPU stays at 0% and the log does not change for about 5 minutes. Stop the watcher cell first. It prints the exact place where the training program is waiting.

In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', 'py-spy'])
result = subprocess.run(['py-spy', 'dump', '--pid', str(process.pid)], capture_output=True, text=True)
print(result.stdout[-4000:])
print(result.stderr[-1000:])
print(open('logs/train.log').read()[-1500:].replace('\r', '\n'))

## 13. Export the splat file

In [ ]:
configs = sorted(glob.glob('outputs/' + scene + '/**/config.yml', recursive=True))
assert configs, 'no config.yml found after training'
config = configs[-1]
run_dir = os.path.dirname(config)
shutil.rmtree('results', ignore_errors=True)
os.makedirs('results', exist_ok=True)
export_started = time.time()
subprocess.run([NS + '/ns-export', 'gaussian-splat', '--load-config', config, '--output-dir', 'results/export'], check=True, env=ENV)
plys = glob.glob('results/export/*.ply')
assert plys, 'export produced no .ply'
ply = plys[0]
export_seconds = int(time.time() - export_started)
print('exported', ply, round(os.path.getsize(ply) / 1e6), 'MB in', export_seconds, 'seconds')

## 14. Score on held-out frames (optional)
If this fails, the result is still valid; it just has no `eval.json`.

In [ ]:
os.makedirs('results/pkg', exist_ok=True)
eval_result = subprocess.run([NS + '/ns-eval', '--load-config', config, '--output-path', 'results/pkg/eval.json'], capture_output=True, text=True, env=ENV)
has_eval = eval_result.returncode == 0 and os.path.exists('results/pkg/eval.json')
print('eval finished' if has_eval else 'WARNING: ns-eval failed, continuing without eval.json')
if not has_eval:
    print(eval_result.stderr[-1500:])

## 15. Pack the result zip
Same file names and contents as the lab PC result, so STEP 3 reads it unchanged.

In [ ]:
package = 'results/pkg'
os.makedirs(package, exist_ok=True)
has_eval = globals().get('has_eval', False) and os.path.exists(package + '/eval.json')
shutil.copyfile(ply, package + '/splat.ply')
shutil.copyfile(config, package + '/config.yml')
shutil.copyfile(run_dir + '/dataparser_transforms.json', package + '/dataparser_transforms.json')
shutil.copyfile('scene.json', package + '/scene.json')
shutil.copyfile('logs/train.log', package + '/step2.log')

gaussians = ''
with open(package + '/splat.ply', 'rb') as handle:
    for line in handle:
        if line.startswith(b'element vertex'):
            gaussians = line.split()[-1].decode()
        if line.startswith(b'end_header'):
            break

timing = [
    'scene: ' + scene,
    'iterations: ' + str(ITERS),
    'downscale: ' + str(DOWNSCALE),
    'cache: ' + CACHE,
    'profile: ' + PROFILE + ' (splatfacto)',
    'train_seconds: ' + str(train_seconds),
    'export_seconds: ' + str(export_seconds),
    'ply_bytes: ' + str(os.path.getsize(package + '/splat.ply')),
    'gaussians: ' + gaussians,
    'gpu: ' + gpu_name,
    'platform: colab',
]
if has_eval:
    scores = json.load(open(package + '/eval.json')).get('results', {})
    for key in ('psnr', 'ssim', 'lpips'):
        if key in scores:
            timing.append('heldout_' + key + ': ' + format(scores[key], '.4f'))
open(package + '/timing.txt', 'w').write('\n'.join(timing) + '\n')

names = ['splat.ply', 'config.yml', 'dataparser_transforms.json', 'scene.json', 'timing.txt', 'step2.log']
if has_eval:
    names.append('eval.json')
zip_path = BASE + '/' + scene + '_' + PROFILE + '_result.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=3) as archive:
    for name in names:
        archive.write(package + '/' + name, name)
print(open(package + '/timing.txt').read())
print('RESULT ZIP:', zip_path, round(os.path.getsize(zip_path) / 1e6), 'MB')
print('modified:', time.ctime(os.path.getmtime(zip_path)))
print('iterations in this zip:', ITERS)

drive_copy = DRIVE_FOLDER + '/' + os.path.basename(zip_path)
shutil.copyfile(zip_path, drive_copy)
print('SAVED TO DRIVE:', drive_copy)

## 16. Get the result onto your laptop
1. Open drive.google.com > the `namma` folder > right-click `<scene>_default_result.zip` > Download.
2. Copy it into `workspace/from_gpu` on the laptop and run STEP 3.

A 2000-step test result only checks that the pipeline works. Do the 30000-step run before you use a result for the website.